# Brickbeck College, University of London

In [1]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

base_url = "https://search.bbk.ac.uk/search/courses/?q="


driver = webdriver.Chrome()

# Function to extract course details from a page
def extract_course_details():
    # Find all elements with course details
    elements = driver.find_elements(By.CLASS_NAME, 'callout--ghost')
    courses_data = []
    for i in range(len(elements)):
        element = elements[i]
        try:
            degree_level = element.find_element(By.CLASS_NAME, 'card__flag__content').text.strip()
        except:
            degree_level="NA"
            
        try:
            course_title = element.find_element(By.CLASS_NAME, 'card__title').text.strip()
        except:
            course_title="NA"
            
        try:
            degree_type = element.find_element(By.CLASS_NAME, 'lead').text.strip()
        except:
            degree_type="NA"
            
        try:
            time_labels = [label.text.strip() for label in element.find_elements(By.CLASS_NAME, 'label--large')]
        except:
            time_labels="NA"
            
        try:
            start_date = element.find_element(By.TAG_NAME, 'time').text.strip()
        except:
            start_date="NA"
            
        try:
            course_link_element = element.find_element(By.CSS_SELECTOR, 'a.row.expanded')
            course_link = course_link_element.get_attribute('href')
        except:
            course_link="NA"

        # Create a dictionary for each course
        course_data = {
            "Course_Title": course_title,
            "Degree_Level": degree_level,
            "Degree_Type": degree_type,
            "Attendance_Mode": ', '.join(time_labels),
            "Start Date": start_date,
            "Course_Link": course_link
        }
        courses_data.append(course_data)

    return courses_data

# Function to check if "Next" button is present
def is_next_button_present():
    try:
        next_button = driver.find_element(By.CLASS_NAME, 'next')
        return next_button.is_displayed() and next_button.is_enabled()
    except:
        return False

# Extract course details from the first page
all_courses = extract_course_details()

# Start iterating through pages
page = 1
while True:
    page_url = f"{base_url}&start_rank={(page - 1) * 20 + 1}"
    driver.get(page_url)
    courses_on_page = extract_course_details()
    all_courses.extend(courses_on_page)
    
    if not is_next_button_present():
        break
    
    page += 1

# Close the browser when done
driver.quit()

# Create a Pandas DataFrame from the list of course details dictionaries
courses_df = pd.DataFrame(all_courses)

# Display the DataFrame
courses_df


,Course_Title,Degree_Level,Degree_Type,Attendance_Mode,Start Date,Course_Link
0,ACCOUNTING,Undergraduate level,BSc (Hons),"Full-time, Part-time, On campus, With or witho...",October 2024,https://birkbeck-search.squiz.cloud/s/redirect...
1,ARCHAEOLOGY,Undergraduate level,BA (Hons),"Full-time, Part-time, On campus, With or witho...",October 2024,https://birkbeck-search.squiz.cloud/s/redirect...
2,BIOMEDICINE,Undergraduate level,BSc (Hons),"Full-time, Part-time, On campus, With or witho...",October 2024,https://birkbeck-search.squiz.cloud/s/redirect...
3,BUSINESS,Undergraduate level,BSc (Hons),"Full-time, Part-time, On campus, With or witho...",October 2024,https://birkbeck-search.squiz.cloud/s/redirect...
4,CLASSICS,Undergraduate level,BA (Hons),"Full-time, Part-time, On campus",October 2024,https://birkbeck-search.squiz.cloud/s/redirect...
...,...,...,...,...,...,...
301,FILM AND SCREEN MEDIA (WITH STUDY ABROAD),Postgraduate level,MA,"Full-time, On campus",October 2024,https://birkbeck-search.squiz.cloud/s/redirect...
302,"FILM, MEDIA AND LANGUAGE (FRENCH, GERMAN, ITAL...",Undergraduate level,BA (Hons),"Full-time, Part-time, On campus, With or witho...",October 2024,https://birkbeck-search.squiz.cloud/s/redirect...
303,"LINGUISTICS AND LANGUAGE (FRENCH, GERMAN, ITAL...",Undergraduate level,CertHE,"Part-time, On campus",October 2024,https://birkbeck-search.squiz.cloud/s/redirect...
304,"LANGUAGE TEACHING AND LANGUAGES (FRENCH, GERMA...",Undergraduate level,BA (Hons),"Full-time, Part-time, On campus, With or witho...",October 2024,https://birkbeck-search.squiz.cloud/s/redirect...


In [2]:
# Save the data to a CSV file
courses_df.to_csv("BrickbeckCollege_UniversityOfLondon.csv", index=False)